Cell 1: Imports

In [1]:
import os
import time
from collections import Counter

import pandas as pd
import requests
from dotenv import load_dotenv
from google import genai

print("Packages loaded")

Packages loaded


Cell 2: Load API keys

In [2]:
# The notebook lives in notebooks/, the .env file is in the project root
load_dotenv("../.env")

gemini_key = os.getenv("GEMINI_API_KEY")
factcheck_key = os.getenv("FACTCHECK_API_KEY")

# Never print the keys themselves: notebook outputs are pushed to GitHub
print("Gemini key loaded:", bool(gemini_key))
print("Fact Check key loaded:", bool(factcheck_key))

Gemini key loaded: True
Fact Check key loaded: True


Cell 3: Load the FCTR dataset

In [3]:
# The file has a .csv extension but is actually tab-separated
df = pd.read_csv("../data/raw/fctr.csv", sep="\t")

print("Shape (rows, columns):", df.shape)
print("Columns:", df.columns.tolist())
print()
print("Label distribution:")
print(df["label"].value_counts())

Shape (rows, columns): (3238, 7)
Columns: ['claim_id', 'claim', 'evidence', 'summary', 'label', 'url', 'date']

Label distribution:
label
yanlış                2780
doğru                  203
karma                  109
kismen yanliş           72
sonuçlandırılamadı      37
yarı doğru              17
çoğunlukla yanlış       14
çoğunlukla doğru         6
Name: count, dtype: int64


Cell 4: Test Gemini

In [4]:
GEMINI_MODEL = "gemini-3.8-flash"

client = genai.Client(api_key=gemini_key)

response = client.models.generate_content(
    model=GEMINI_MODEL,
    contents="RAG nedir? Türkçe, tek cümleyle açıkla."
)
print(response.text)

**RAG (Geri Getirme ile Artırılmış Üretim)**, yapay zekâ dil modellerinin yanıt üretmeden önce harici bilgi kaynaklarından ilgili verileri çekerek daha doğru, güncel ve bağlama uygun cevaplar vermesini sağlayan bir yöntemdir.


Cell 5: Request helper with retries

In [5]:
FACTCHECK_URL = "https://factchecktools.googleapis.com/v1alpha1/claims:search"


def send_request(params, max_retries=4):
    """Send a GET request to the Fact Check API, retrying on temporary errors."""
    for attempt in range(max_retries):
        response = requests.get(FACTCHECK_URL, params=params)

        if response.status_code == 200:
            return response

        # Retry only on temporary errors; 400/403 will not fix themselves
        if response.status_code in (429, 500, 503):
            wait = 2 ** attempt  # Exponential backoff: 1, 2, 4, 8 seconds
            print(f"  Got {response.status_code}, retrying in {wait}s...")
            time.sleep(wait)
            continue

        return response

    return response

Cell 6: Latest fact-checks per publisher

In [7]:
def search_by_publisher(site, page_size=5):
    """Print the latest Turkish fact-checks from a given publisher."""
    params = {
        "reviewPublisherSiteFilter": site,
        "languageCode": "tr",
        "pageSize": page_size,
        "key": factcheck_key,
    }
    response = send_request(params)
    print(f"\n===== {site} | Status: {response.status_code} =====")

    if response.status_code != 200:
        print("Error:", response.text[:300])
        return

    claims = response.json().get("claims", [])
    print("Results:", len(claims))

    for claim in claims:
        review = claim["claimReview"][0]
        print("Claim :", claim.get("text"))
        print("Rating:", review.get("textualRating"))
        print("Date  :", review.get("reviewDate"))
        print("URL   :", review.get("url"))
        print("-" * 60)


for site in ["teyit.org", "dogrulukpayi.com", "dogrula.org"]:
    search_by_publisher(site)


===== teyit.org | Status: 200 =====
Results: 5
Claim : Kuzey Kore'de Kur'an'a hakaret yasaklandı mı?
Rating: Yanlış
Date  : 2026-10-17T00:00:00Z
URL   : https://teyit.org/analiz/kuzey-korede-kurana-hakaret-yasaklandi-mi
------------------------------------------------------------
Claim : İstanbul'da bir uçağın havalimanında savrulduğunu gösteren görüntüler gerçek mi?
Rating: Yanlış
Date  : 2026-10-16T00:00:00Z
URL   : https://teyit.org/analiz/istanbulda-bir-ucagin-havalimaninda-savruldugunu-gosteren-goruntuler-gercek-mi
------------------------------------------------------------
Claim : Sel felaketinden sonra sigorta şirketlerinin iflas ettiği iddiası
Rating: Yanlış
Date  : 2026-10-16T00:00:00Z
URL   : https://teyit.org/analiz/sel-felaketinden-sonra-sigorta-sirketlerinin-iflas-ettigi-iddiasi
------------------------------------------------------------
Claim : Kar görüntülerinin Ankara'dan olduğu iddiası
Rating: Yanlış
Date  : 2026-10-16T00:00:00Z
URL   : https://teyit.org/analiz/kar-

Cell 7: Discover Turkish publishers available in the API

In [8]:
def discover_publishers(queries, page_size=50):
    """Count which publishers appear in Turkish search results."""
    counts = Counter()

    for query in queries:
        params = {
            "query": query,
            "languageCode": "tr",
            "pageSize": page_size,
            "key": factcheck_key,
        }
        response = send_request(params)
        if response.status_code != 200:
            print(f"{query} -> skipped, status {response.status_code}")
            continue

        for claim in response.json().get("claims", []):
            for review in claim.get("claimReview", []):
                site = review.get("publisher", {}).get("site", "unknown")
                counts[site] += 1

        time.sleep(1)  # Be polite to the API between requests

    return counts


# Queries stay in Turkish because we are searching Turkish content
queries = ["video", "deprem", "aşı", "seçim", "ekonomi", "sel",
           "fotoğraf", "iddia", "sahte", "yapay zeka"]

publisher_counts = discover_publishers(queries)
for site, count in publisher_counts.most_common():
    print(f"{site:35} {count}")

dogrulukpayi.com                    230
teyit.org                           195
dogrula.org                         11
cancerdisinfo.org                   1
